# End of week 1 exercise

To demonstrate your familiarity with OpenAI API, and also Ollama, build a tool that takes a technical question,  
and responds with an explanation. This is a tool that you will be able to use yourself during the course!

In [1]:
# imports
import os
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import display, Markdown, update_display

In [2]:
# constants

MODEL_GEMINI = 'gemini-3.1-flash-lite'
MODEL_LLAMA = 'llama3.2:latest'

In [3]:
# set up environment
load_dotenv(override=True)

#setup model gemini
GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"
gemini = OpenAI(base_url=GEMINI_BASE_URL, api_key=os.environ.get("GOOGLE_API_KEY"))

#setup model llama
LLAMA_BASE_URL = "http://localhost:11434/v1"
ollama = OpenAI(base_url=LLAMA_BASE_URL, api_key="ollama")


In [4]:
# here is the question; type over this to ask something new

question = """
Please explain what this code does and why:
yield from {book.get("author") for book in books if book.get("author")}
"""

In [5]:
# Get gemini-3.1-flash-lite to answer, with streaming
stream = gemini.chat.completions.create(
    model=MODEL_GEMINI,
    messages=[
        {"role" : "system", "content" : "You are senior python developer. Explain the code clearly and concisely in markdown and use bahasa indonesia."},
        {"role": "user", "content": question}
    ],
    stream=True
)

response = ""
display_handle =display(Markdown(""), display_id=True)

for chunk in stream:
    response += chunk.choices[0].delta.content or ""
    update_display(Markdown(response), display_id=display_handle.display_id)

Kode ini adalah cara yang sangat ringkas (idiomatik) dalam Python untuk **mengekstrak daftar nama penulis yang unik dari sebuah koleksi buku.**

Berikut adalah penjelasan detailnya:

### Penjelasan Bagian per Bagian

1.  **`{book.get("author") for book in books if book.get("author")}`**:
    *   Ini adalah sebuah **Set Comprehension**.
    *   `for book in books`: Melakukan iterasi (perulangan) pada setiap objek/dictionary `book` di dalam list `books`.
    *   `if book.get("author")`: Filter agar hanya memproses buku yang memiliki kunci `"author"` dan nilainya tidak kosong (truthy).
    *   `book.get("author")`: Mengambil nilai dari kunci `"author"`.
    *   **Hasilnya:** Karena menggunakan kurung kurawal `{}`, Python akan membuat sebuah **`set`** (himpunan). Keunggulan `set` adalah secara otomatis **menghilangkan duplikasi**. Jadi, jika ada 5 buku dengan penulis yang sama, nama penulis tersebut hanya akan muncul satu kali.

2.  **`yield from`**:
    *   Ini adalah delegasi generator. Jika kode ini berada di dalam sebuah *generator function*, `yield from` akan melakukan iterasi pada hasil set tadi dan mengeluarkan (yield) setiap elemennya satu per satu ke pemanggil fungsi.

---

### Mengapa Menggunakan Kode Ini?

*   **Efisien dalam memori:** Karena menggunakan generator (`yield from`), Anda tidak perlu membuat list besar di memori. Data dikirim satu per satu saat dibutuhkan.
*   **Menghilangkan Duplikat:** Penggunaan `{}` (set) adalah cara paling cepat dan bersih untuk memastikan setiap nama penulis hanya muncul sekali tanpa harus melakukan pengecekan manual (`if name not in list`).
*   **Safe Access:** Penggunaan `.get("author")` mencegah program *crash* (KeyError) jika ada objek buku yang tidak memiliki kunci `"author"`.
*   **Keterbacaan:** Menggabungkan logika filtering, ekstraksi, dan penghilangan duplikat ke dalam satu baris kode yang sangat deklaratif.

### Contoh Skenario

```python
books = [
    {"title": "Buku A", "author": "Budi"},
    {"title": "Buku B", "author": "Ani"},
    {"title": "Buku C", "author": "Budi"}, # Duplikat
    {"title": "Buku D"}                    # Tidak ada author
]

def get_unique_authors(books):
    yield from {book.get("author") for book in books if book.get("author")}

# Output: 'Budi', 'Ani' (urutan bisa acak karena set tidak terurut)
print(list(get_unique_authors(books)))
```

**Kesimpulan:** Baris kode ini adalah cara "Pythonic" untuk mendapatkan daftar unik dari sebuah properti objek di dalam list dengan performa yang optimal.

In [6]:
# Get Llama 3.2 to answer
response_llama = ollama.chat.completions.create(
    model=MODEL_LLAMA,
    messages=[
        {"role": "system", "content": "You are a senior Python developer. Explain the code clearly and concisely in markdown and use bahasa indonesia"},
        {"role": "user", "content": question}
    ]
)

display(Markdown(response_llama.choices[0].message.content))

**Penjelasan Code di Atas**
==========================

Code yang kita lihat di atas menggunakan sintaks penggunaan generator di Python. Berikut adalah penjelasannya:

```python
# yield from adalah sebuah operator penghubung antara dua lingkaran generator.
yield from {book.get("author") for book in books if book.get("author")}
```

**Apa yang dibuatilah?**
--------------------

Code tersebut membuat sebuah perulangan generator (perulangan tanpa membuat semuanya dalam RAM sekaligus) untuk membuat daftar penulis buku.

Berikut adalah apa-apa yang terjadi di dalam code ini:

*   `{book.get("author")}`: membuat setiap satu buku pada daftar `books` yang memiliki data "penulis" akan diambil ke dalam `set`.
*   `for book in books`: untuk setiap satu buku di dalam daftar, gunakan pengulangan untuk memberikan penulis dari book tersebut pada perulangan generator. Pengembalian dari setiap ulang (book) dilakukan saat kita "yield from" nilai yang diambil ke dalam dictionary.
*   `if book.get("author")`: akan mengeksekusi apakah daftar buku `book` punya data penulis atau tidak, jika benar terus melanjutkannya untuk memasukkan nilai tersebut pada perulangan generator.

**Tujuan Akhir:**
-----------------

Tujuan dari code di atas adalah mengambil semua penulis dari daftar buku ke dalam satu generator, yang kemudian dapat diproses jika kita punya program untuk melakukan hal ini.